In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, KFold, GridSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
import shap
import seaborn as sns

/Users/andrearaaschou/courses/BERN01/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load data 
lpj_df = pd.read_csv("../data/LPJ-GUESS_output_BERN1.csv")
grid_df = pd.read_csv("../data/gridlist_pan_gfed_ISO3_UN_ecoregion.txt",sep=" ")
soil_df = pd.read_csv("../data/soilmap.dat",sep="\t")

# Load all climate datafiles and put in one dictionary
climate_files = {
    "tswrf": "../data/Tswrfdaymean1961_1990.csv",
    "pre": "../data/Predaymean1961_1990.csv",
    "tmp": "../data/Tmpdaymean1961_1990.csv",
    "tmax": "../data/Tmaxdaymean1961_1990.csv",
    "tmin": "../data/Tmindaymean1961_1990.csv"}
climate_df = {}
for name, path in climate_files.items():
    climate_df[name] = pd.read_csv(path)

# Load all climate statistics datafiles and put in one dictionary
stats_files = {
    "tswrf": "../data/Tswrfdaymean1961_1990_stats.csv",
    "pre": "../data/Predaymean1961_1990_stats.csv",
    "tmp": "../data/Tmpdaymean1961_1990_stats.csv",
    "tmax": "../data/Tmaxdaymean1961_1990_stats.csv",
    "tmin": "../data/Tmindaymean1961_1990_stats.csv"}
stats_df = {}
for name, path in stats_files.items():
    stats_df[name] = pd.read_csv(path)

In [3]:
# Fix problems with soil_df
soil_df = soil_df.dropna() # drops all rows with nan values
soil_df[soil_df["lon"].isna()]
soil_df["lon"] = pd.to_numeric(soil_df["lon"]) # Change the lon column values from type str to float
soil_df.shape # check how many rows were dropped
soil_df = soil_df.rename(columns={"lon": "Lon","lat": "Lat"}) # Rename columns

In [ ]:
# Merge df to one collected_df
collected_df = lpj_df[["Lon", "Lat", "Biome_obs"]].copy() # Start by taking wanted columns from lpj_df
collected_df = collected_df.merge(soil_df, on=["Lon", "Lat"], how="left") # Merge with soil_df
for name, df in stats_df.items(): # Merge with stats (climate) as well
    collected_df = collected_df.merge(
        df,
        on=["Lon", "Lat"],
        how="left",
        suffixes=("", f"_{name}") # add suffixes to keep climate columns apart
    )